In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import os
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"

tf.config.threading.set_intra_op_parallelism_threads(6)
tf.config.threading.set_inter_op_parallelism_threads(2)

# Load mô hình và path folder toàn ảnh để giải đoán

model = load_model("results/CNNmodel.keras")
    
folder_path = "data/alos_images"

In [2]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

img_files = sorted([
    os.path.join(folder_path, f)
    for f in os.listdir(folder_path)
    if f.endswith(".img")
])

bands = []
ref_profile = None

# Ảnh chuẩn (đầu tiên)
with rasterio.open(img_files[0]) as ref:
    ref_profile = ref.profile
    ref_data = ref.read(1)
    transform = ref.transform
    crs = ref.crs
    nodata = ref.nodata if ref.nodata is not None else 0

bands.append(ref_data)

# Căn chỉnh các ảnh còn lại theo ref
for path in img_files[1:]:
    with rasterio.open(path) as src:
        band = np.zeros((ref.height, ref.width), dtype=src.read(1).dtype)
        reproject(
            source=src.read(1),
            destination=band,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref.transform,
            dst_crs=ref.crs,
            resampling=Resampling.nearest
        )
        bands.append(band)

# Stack lại sau khi căn chỉnh
img = np.stack(bands, axis=-1)
print("Ảnh sau khi căn chỉnh:", img.shape)


Ảnh sau khi căn chỉnh: (8252, 8282, 4)


In [3]:
from tqdm import tqdm

# --- Cấu hình patch ---
patch_size = 24
stride = 6

# img = ảnh 3D (H x W x bands), src = rasterio source ban đầu
height, width, n_bands = img.shape

# --- Tính padding ---
pad_h = (patch_size - height % patch_size) % patch_size
pad_w = (patch_size - width % patch_size) % patch_size

# --- Padding ảnh ---
img_pad = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')

H_pad, W_pad, _ = img_pad.shape

# --- Gom patch thành batch ---
patches = []
positions = []

print("🔍 Đang gom patch từ ảnh...")

for i in range(0, H_pad - patch_size + 1, stride):
    for j in range(0, W_pad - patch_size + 1, stride):
        patch = img_pad[i:i+patch_size, j:j+patch_size, :]
        patches.append(patch)
        positions.append((i, j))

patches = np.array(patches, dtype=np.float32)
num_patches = len(patches)

print(f"📦 Tổng số patch cần predict: {num_patches:,}")

# --- Predict theo batch lớn ---
batch_size = 512   # có thể tăng 1024 nếu RAM còn trống

print("🚀 Đang predict theo batch...")
preds = model.predict(patches, batch_size=batch_size, verbose=1)
pred_classes = np.argmax(preds, axis=1)

# --- Gán kết quả vào raster ---
classified = np.zeros((H_pad, W_pad), dtype=np.uint8)

print("🧩 Đang ghép kết quả vào ma trận phân loại...")

for (i, j), cls in zip(positions, pred_classes):
    classified[i:i+stride, j:j+stride] = cls

# --- Cắt padding về kích thước gốc ---
classified = classified[:height, :width]

# --- Lưu TIFF kết quả ---
output_raster = "results/outputimage.tif"

meta = src.meta.copy()
meta.update({
    "count": 1,
    "dtype": rasterio.uint8,
    "driver": "GTiff",
    "nodata": 255
})

print("💾 Đang lưu raster kết quả...")

with rasterio.open(output_raster, 'w', **meta) as dst:
    dst.write(classified.astype(np.uint8), 1)

print("✅ Lưu xong ảnh phân loại tại:", output_raster)


🔍 Đang gom patch từ ảnh...


📦 Tổng số patch cần predict: 1,896,113
🚀 Đang predict theo batch...


   1/3704 ━━━━━━━━━━━━━━━━━━━━ 1:10:31 1s/step

   2/3704 ━━━━━━━━━━━━━━━━━━━━ 4:49 78ms/step 

   4/3704 ━━━━━━━━━━━━━━━━━━━━ 3:12 52ms/step

...

3704/3704 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step

3704/3704 ━━━━━━━━━━━━━━━━━━━━ 102s 27ms/step


🧩 Đang ghép kết quả vào ma trận phân loại...


💾 Đang lưu raster kết quả...


✅ Lưu xong ảnh phân loại tại: G:/CatTuong/ALOS_Processing/classification_output/CNNresult/CNN2025_part2_new.tif
